## init

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set all fonts to Arial size 7
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 7,
    'axes.titlesize': 7,
    'axes.labelsize': 7,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7
})

In [ ]:
from autoadsorbate import Surface
from ase.io import read, write
from ase.visualize import view
import numpy as np
from ase import Atoms
import random
import math

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.patches import Rectangle
from matplotlib.ticker import FixedLocator


from cft import Manifold
from autoadsorbate.Particle import get_cube_surface_pts, grid_round_cube
from cft.mesh_utils import compute_outward_vertex_normals_quads

from ase.io import read, write
from ase.visualize import view
from autoadsorbate import Fragment
from autoadsorbate.Surf import attach_fragment
from ase.constraints import FixAtoms
import copy

In [4]:
from mace.calculators import mace_mp
clean_calc = mace_mp(model=
                # '/mnt/c/Users/ef/Desktop/tmp/mace-mh-nl-pbe.model',
                '/mnt/c/Users/ef/Desktop/tmp/mace-omat-0-medium.model',
                device='cpu',
                )

/home/ef/venvs/mace_env/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.


/home/ef/venvs/mace_env/lib/python3.12/site-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Using head default out of ['default']
Default dtype float32 does not match model dtype float64, converting models to float32.


In [139]:
p = .1
c = 20
scale = [5,5,2]
bulk = read('./TiN.cif')


slab = bulk.copy()*scale
slab.cell[2][2] += c
slab.positions[:,2] += c/2
slab.arrays['fragments'] = np.array([0 for _ in slab])

#make some vacancies
_s = Surface(slab)
n_inds = [atom.index for atom in _s.atoms if atom.symbol =='N' and atom.index in _s.surf_inds]
n_vac = random.sample(n_inds, math.ceil(p * len(n_inds)))
slab = slab[[atom.index for atom in slab if atom.index not in n_vac]]

# al_z = slab.positions[np.where(slab.positions[:,2] == np.max(slab.positions[:,2]))[0][0]][2]
# al_pos = slab.cell[0]*0.5+slab.cell[1]*0.5 + [0,0,al_z]

# for x in range(3):
#     for y in range(3):
#         for z in range(1,3):
#             print(x,y,z)
#             slab += Atoms(['Al'], [al_pos+[x*2,y*2,z*2]])

slab.set_constraint(FixAtoms(indices=[atom.index for atom in slab if atom.position[2] < slab.cell[2][2]*.5]))
slab.rattle(stdev=.2)

view(slab)

<Popen: returncode: None args: ['/home/ef/venvs/mace_env/bin/python', '-m', ...>

In [137]:
from ase.optimize import BFGS

slab.calc = clean_calc
opt = BFGS(slab, trajectory='relax.xyz')
opt.run(fmax=0.1)

      Step     Time          Energy          fmax
BFGS:    0 16:04:11     -529.229919       55.548660
BFGS:    1 16:04:11     -567.652405       26.572554
BFGS:    2 16:04:12     -601.993713       17.629202
BFGS:    3 16:04:12     -624.147522       11.004293
BFGS:    4 16:04:13     -639.933899        6.598811
BFGS:    5 16:04:13     -649.778015        3.779891
BFGS:    6 16:04:14     -655.264221        2.507646
BFGS:    7 16:04:14     -658.159485        2.778377
BFGS:    8 16:04:15     -659.675903        2.103954
BFGS:    9 16:04:15     -660.882751        1.850347
BFGS:   10 16:04:16     -661.669556        1.972395
BFGS:   11 16:04:16     -662.327026        1.639348
BFGS:   12 16:04:17     -662.786926        1.168407
BFGS:   13 16:04:18     -663.245239        1.041232
BFGS:   14 16:04:18     -663.693481        1.039846
BFGS:   15 16:04:19     -664.043884        0.947292
BFGS:   16 16:04:19     -664.303528        0.786271
BFGS:   17 16:04:20     -664.522644        0.701764
BFGS:   18 16:

True

In [138]:
view(slab)

<Popen: returncode: None args: ['/home/ef/venvs/mace_env/bin/python', '-m', ...>

## manifold

User requested to_initialize = 1 conformers.
After pruning with 0.0001; len(conformer_trj) = 1 unique conformers are found.


[15:28:00] UFFTYPER: Unrecognized charge state for atom: 1
[15:28:00] UFFTYPER: Unrecognized charge state for atom: 1


In [ ]:
def make_custom_probe_scan(slab):
    f = Fragment('Cl[P+](C)(C)C', to_initialize=1)
    for atoms in f.conformers:
        for atom in atoms:
            if atom.symbol =='P':
                atom.symbol ='Al'

    m = Manifold(slab,
                precision=1,
                touch_sphere_size =2.5,
                wrap_on='atoms',
                calc = clean_calc)
    m.normals*=-1

    m.run_probe_scan(probes=[f, Fragment('ClC', to_initialize=1), (Atoms(['Al'], [0,0,0]))])
    m.write_grid('grd.xyz')


In [111]:
m.write_grid('grd.xyz')

In [170]:
# xx = read('/mnt/c/Users/ef/Desktop/tmp/grd_run.xyz', index=0)

## population

In [171]:
slab = read('/mnt/c/Users/ef/Desktop/tmp/relax_run.xyz', index=-1)
view(slab)

<Popen: returncode: None args: ['/home/ef/venvs/mace_env/bin/python', '-m', ...>

In [172]:
write('/mnt/c/Users/ef/Desktop/tmp/relax_run_slab-1.xyz', slab)

In [ ]:
f = Fragment('Cl[P+](C)(C)C', to_initialize=100)
for atoms in f.conformers:
    for atom in atoms:
        if atom.symbol =='P':
            atom.symbol ='Al'

m = Manifold(slab,
                precision=1,
                touch_sphere_size =2.5,
                wrap_on='atoms',
                calc = clean_calc)

m.normals*=-1


pop_traj = []

for coverage in [.2,.5,.6,.7,.8,.9,1.]:
    m.make_fragment_population(
                population_size = 10,
                fragment = f,
                coverage = coverage
                )
    pop_traj+=m.surf_population

view(pop_traj)

[17:30:49] UFFTYPER: Unrecognized charge state for atom: 1
[17:30:49] UFFTYPER: Unrecognized charge state for atom: 1


User requested to_initialize = 100 conformers.
After pruning with 0.5; len(conformer_trj) = 1 unique conformers are found.


<Popen: returncode: None args: ['/home/ef/venvs/mace_env/bin/python', '-m', ...>

In [166]:
for a in pop_traj:
    a.arrays['fragments'] += (a.arrays['fragments'] > 0) * 10

In [167]:
write('/mnt/c/Users/ef/Desktop/tmp/pop_traj.xyz', pop_traj)

In [1]:
# pop_traj = read('/mnt/c/Users/ef/Desktop/tmp/pop_traj.xyz', index = ':')
# for i, atoms in enumerate(pop_traj):
#     atoms.set_constraint(FixAtoms(indices=[atom.index for atom in atoms if atom.symbol in ['N', 'Ti']]))
#     atoms.calc = clean_calc
#     opt = BFGS(atoms, trajectory='tmp.xyz')
#     print(f'{i = }')
#     opt.run(fmax=0.5)

In [153]:
m.surf_population[0].arrays.keys()

dict_keys(['numbers', 'positions', 'spacegroup_kinds', 'fragments'])

In [ ]:
for atoms in m.surf_population:


m.evaluate_surf_population()

Calculating energies: 100%|██████████| 1/1 [00:00<00:00,  2.15it/s]
